In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import date_trunc, count, countDistinct


@dp.table(
    name="gharchive_gold_activity_counts",
    comment="Hourly GitHub event counts by event type"
)
def gharchive_gold_activity_counts():
    # dp.read() automatically registers dependency on gharchive_silver
    df = dp.read("gharchive_silver")

    return (
        df
        .withColumn("event_hour", date_trunc("hour", "created_at"))
        .groupBy("event_hour", "type")
        .agg(
            count("*").alias("event_count"),
            countDistinct("actor_login").alias("unique_actors"),
            countDistinct("repo_name").alias("unique_repos")
        )
        .orderBy("event_hour", "type")
    )

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import count, countDistinct


@dp.table(
    name="gharchive_gold_top_repos",
    comment="Top repositories ranked by total event count"
)
def gharchive_gold_top_repos():
    # dp.read() auto-registers dependency on silver
    df = dp.read("gharchive_silver")

    return (
        df
        .groupBy("repo_name")
        .agg(
            count("*").alias("total_events"),
            countDistinct("actor_login").alias("unique_contributors"),
            countDistinct("type").alias("event_types")
        )
        .orderBy(count("*").desc())
    )

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import count, countDistinct, min, max


@dp.table(
    name="gharchive_gold_top_actors",
    comment="Top GitHub actors ranked by contribution count"
)
def gharchive_gold_top_actors():
    # dp.read() auto-registers dependency on silver
    df = dp.read("gharchive_silver")

    return (
        df
        .groupBy("actor_login")
        .agg(
            count("*").alias("total_events"),
            countDistinct("repo_name").alias("unique_repos"),
            countDistinct("type").alias("event_types"),
            min("created_at").alias("first_event"),
            max("created_at").alias("last_event")
        )
        .orderBy(count("*").desc())
    )